In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense

In [2]:
df = pd.read_csv("Tweets.csv")

In [3]:
df.shape

(14640, 15)

In [4]:
df.isnull().sum()

,0
tweet_id,0
airline_sentiment,0
airline_sentiment_confidence,0
negativereason,5462
negativereason_confidence,4118
airline,0
airline_sentiment_gold,14600
name,0
negativereason_gold,14608
retweet_count,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null  object 
 13  t

In [6]:
df = df[["text", "airline_sentiment"]].dropna()

In [7]:
le = LabelEncoder()
y = le.fit_transform(df["airline_sentiment"])

print("Classes:", le.classes_)

Classes: ['negative' 'neutral' 'positive']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [10]:
len(X_train_seq[1])

28

In [11]:
len(X_train_seq[1230])

21

In [12]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=50,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=50,
    padding="post",
    truncating="post"
)

In [14]:
n_classes = len(np.unique(y))


In [15]:
model = Sequential([
    Embedding(5000, 32, input_length=50),
    LSTM(32),
    Dense(n_classes, activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
# 10. Compile model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [17]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.6257 - loss: 0.9139 - val_accuracy: 0.5809 - val_loss: 0.8552
Epoch 2/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.7337 - loss: 0.6351 - val_accuracy: 0.7725 - val_loss: 0.5730
Epoch 3/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.8110 - loss: 0.4830 - val_accuracy: 0.7742 - val_loss: 0.5578
Epoch 4/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.8497 - loss: 0.4044 - val_accuracy: 0.7456 - val_loss: 0.6308
Epoch 5/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.8723 - loss: 0.3584 - val_accuracy: 0.7405 - val_loss: 0.6410


In [18]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("\nLSTM Test Accuracy:", test_accuracy)


LSTM Test Accuracy: 0.7380464673042297


In [19]:
y_pred = np.argmax(model.predict(X_test_pad), axis=1)

print("Accuracy Check:", accuracy_score(y_test, y_pred))


92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step
Accuracy Check: 0.7380464480874317


GRU Model

In [20]:
gru_model = Sequential([
    Embedding(5000, 32, input_length=50),
    GRU(32),
    Dense(n_classes, activation="softmax")
])

gru_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gru_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

gru_loss, gru_accuracy = gru_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("\nGRU Test Accuracy:", gru_accuracy)

Epoch 1/5


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


293/293 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - accuracy: 0.6278 - loss: 0.9216 - val_accuracy: 0.6236 - val_loss: 0.9194
Epoch 2/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.6278 - loss: 0.9170 - val_accuracy: 0.6236 - val_loss: 0.9200
Epoch 3/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.6278 - loss: 0.9167 - val_accuracy: 0.6236 - val_loss: 0.9204
Epoch 4/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 16s 55ms/step - accuracy: 0.6637 - loss: 0.7794 - val_accuracy: 0.7192 - val_loss: 0.6338
Epoch 5/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.7754 - loss: 0.5389 - val_accuracy: 0.7550 - val_loss: 0.6104

GRU Test Accuracy: 0.7592213153839111


The forget gate allows an LSTM to decide which old information should be kept and which information should be discarded from its cell state. A plain RNN does not have this controlled memory mechanism, so it struggles more with long-term dependencies and vanishing gradients.